### ch3 Functional data structures

In [1]:
sealed class List<out A> {
    companion object {
        operator fun <A> invoke(vararg aa: A): List<A> =
            if (aa.isEmpty())
                Nil
            else
                Cons(aa[0], List(*aa.sliceArray(1 .. aa.size - 1)))

        fun <A> empty(): List<A> =
            Nil
    }
}

object Nil: List<Nothing>() {
    override fun toString(): String = "Nil"
}

data class Cons<out A>(val head: A, val tail: List<A>) : List<A>() {
    override fun toString(): String =
        "$head :: ${tail.toString()}"
}

In [2]:
val ex1 = Nil
val ex2 = Cons(1, Nil)
val ex3 = Cons("a", Cons("b", Nil))

//val cons = List(1, 2)
//val nil = List<Int>()

In [3]:
fun sum(ints: List<Int>): Int =
    when (ints) {
        is Nil -> 0
        is Cons -> ints.head + sum(ints.tail)
        else -> error("")
    }

fun product(doubles: List<Double>): Double {
    tailrec fun loop(doubles: List<Double>, acc: Double): Double =
        when (doubles) {
            is Nil -> acc
            is Cons ->
                if (doubles.head == 0.0) 0.0
                else loop(doubles.tail, doubles.head * acc)
            else -> error("")
        }
    return loop(doubles, 1.0)
}

product(List(1.0, 2.0, 3.0))


6.0

In [4]:
import kotlin.random.Random

val x = Random.nextInt(-10, 10)
val y = when {
    x == 0 -> "x is zero"
    x < 0 -> "x is negative"
    else -> "x is positive"
}
y

x is zero

#### ex 3.1

In [5]:
fun <A> tail(xs: List<A>): List<A> =
    when (xs) {
        is Nil -> error("Nil")
        is Cons -> xs.tail
        else -> error("")
    }

#### ex 3.2

In [6]:
fun <A> setHead(xs: List<A>, x: A): List<A> =
    when (xs) {
        is Nil -> error("Nil")
        is Cons -> Cons(x, xs.tail)
        else -> error("")
    }

setHead(List(1, 2, 3), 10)

10 :: 2 :: 3 :: Nil

#### ex 3.3

In [7]:
tailrec fun <A> drop(xs: List<A>, n: Int): List<A> =
    if (n == 0)
        xs
    else
        when (xs) {
            is Nil -> error("")
            is Cons -> drop(xs.tail, n - 1)
            else -> error("")
        }

drop(List(1, 2, 3), 2)

3 :: Nil

#### 3.4

In [8]:
fun <A> dropWhile(l: List<A>, f: (A) -> Boolean): List<A> =
    when (l) {
        is Nil -> Nil
        is Cons -> if (f(l.head)) dropWhile(l.tail, f) else l
        else -> error("")
    }

dropWhile(List(1, 3, 1, 2, 3)) { it % 2 == 1 }

2 :: 3 :: Nil

In [9]:
fun <A> append(a1: List<A>, a2: List<A>): List<A> =
    when (a1) {
        is Nil -> a2
        is Cons -> Cons(a1.head, append(a1.tail, a2))
        else -> error("")
    }

#### ex 3.5

In [10]:
fun <A> init(l: List<A>): List<A> =
    when (l) {
        is Nil -> error("Nil")
        is Cons -> when (l.tail) {
            is Nil -> Nil
            is Cons -> Cons(l.head, init(l.tail))
            else -> error("")
        }
        else -> error("")
    }


init(List(1, 2, 3))


1 :: 2 :: Nil

In [11]:
fun <A, B> foldRight(xs: List<A>, z: B, f: (A, B) -> B): B =
    when (xs) {
        is Nil -> z
        is Cons -> f(xs.head, foldRight(xs.tail, z, f))
        else -> error("")
    }
foldRight(List(1, 2, 3), 0) { a, b -> a + b }
// (1 + (2 + (3 + 0)))


//foldRight(List(3, 6, 9), 1.0) { a, b -> a * b }
// foldRight(List(1, 2, 3), 0) { a, b -> a - b }


6

#### ex 3.6

In [12]:
// 모든 식이 확장되어 하나씩 평가되므로 중간에 빠져 나오는건 불가능
// f 에서 foldRight 를 빠져나올 수 있는 goto 가 필요하다. 없음

#### ex 3.7

In [13]:
// list 를 복사한다

#### ex 3.8

In [14]:
fun <A> length(xs: List<A>): Int =
    foldRight(xs, 0) { _, l -> l + 1 }

length(List(1, 2, 3))

3

#### ex 3.9

In [15]:
tailrec fun <A, B> foldLeft(xs: List<A>, z: B, f: (B, A) -> B): B =
    when (xs) {
        is Nil -> z
        is Cons -> foldLeft(xs.tail, f(z, xs.head), f)
        else -> error("")
    }

// ((1 * 2) * 3) * 4
println(foldLeft(List(2, 3, 4), 1) { n, acc -> n * acc })

24


#### ex 3.10

In [16]:
fun sum(l: List<Int>): Int =
    foldLeft(l, 0) { acc, n -> acc + n }

println(sum(List(3, 4, 5)))

fun product(l: List<Double>): Double =
    foldLeft(l, 1.0) { acc, n -> acc * n }

println(product(List(4.0, 5.0, 6.0)))

12
120.0


#### ex 3.11

In [17]:
//fun <A> reverse(l: List<A>): List<A> =
//    when (l) {
//        is Nil -> Nil
//        is Cons -> append(reverse(l.tail), Cons(l.head, Nil))
//        else -> error("")
//    }

fun <A> reverse(l: List<A>): List<A> =
    foldLeft(l, List.empty()) { acc, i -> Cons(i, acc) }

println(reverse<Int>(List(1, 2, 3)))

3 :: 2 :: 1 :: Nil


#### ex 3.12 (hard)

In [18]:
fun <A, B> foldRightA(xs: List<A>, z: B, f: (A, B) -> B): B =
    foldLeft(xs, { b: B -> b }) { acc, a ->
        { b: B -> acc(f(a, b)) }
    }(z)

// foldRightA(List(1, 2, 3), Nil) { a, acc -> Cons(a, acc) }

// foldLeft(List(1, 2, 3), { b0 -> b0 }, { acc, a -> { b -> acc({ a0, acc0 -> Cons(a0, acc0) }(a, b)) } })(Nil)
// foldLeft(List(1, 2, 3), { b0 -> b0 }, { acc, a -> { b -> acc(Cons(a, b)) } })(Nil)

// foldLeft(List(2, 3), { b1 -> { b0 -> b0 }(Cons(1, b1)), { acc, a -> { b -> acc(Cons(a, b)) } })(Nil)
// foldLeft(List(2, 3), { b1 -> Cons(1, b1) }, { acc, a -> { b -> acc(Cons(a, b)) } })(Nil)

// foldLeft(List(3), { acc, a -> { b2 -> acc(Cons(a, b2)) } }({ b1 -> Cons(1, b1) }, 2), { acc, a -> { b -> acc(Cons(a, b)) } })(Nil)
// foldLeft(List(3), { b2 -> { b1 -> Cons(1, b1) }(Cons(2, b2)) } }, { acc, a -> { b -> acc(Cons(a, b)) } })(Nil)

// foldLeft(Nil, { acc, a -> { b3 -> acc(Cons(a, b3)) } }({ b2 -> { b1 -> Cons(1, b1) }(Cons(2, b2)) } }, 3), { acc, a -> { b -> acc(Cons(a, b)) } })(Nil)
// foldLeft(Nil, { b3 -> { b2 -> { b1 -> Cons(1, b1) }(Cons(2, b2)) } }(Cons(3, b3)) }, { acc, a -> { b -> acc(Cons(a, b)) } })(Nil)

// { b3 -> { b2 -> { b1 -> Cons(1, b1) }(Cons(2, b2)) } }(Cons(3, b3)) }(Nil)
// { b2 -> { b1 -> Cons(1, b1) }(Cons(2, b2)) } }(Cons(3, Nil))
// { b1 -> Cons(1, b1) }(Cons(2, Cons(3, Nil))) }
// Cons(1, Cons(2, Cons(3, Nil)))
// List(1, 2, 3)

println(foldRight(List(1, 2, 3), List.empty<Int>()) { a, acc -> Cons(a, acc) })
println(foldRightA(List(1, 2, 3), List.empty<Int>()) { a, acc -> Cons(a, acc) })



1 :: 2 :: 3 :: Nil
1 :: 2 :: 3 :: Nil


#### ex 3.13

In [19]:
fun <A> append1(l1: List<A>, l2: List<A>): List<A> =
    foldRight(l1, l2) { a, acc -> Cons(a, acc) }

println(append1(List(1, 2), List(3, 4)))

fun <A> append2(l1: List<A>, l2: List<A>): List<A> =
    foldLeft(reverse(l1), l2) { acc, a -> Cons(a, acc) }

println(append2(List(1, 2), List(3, 4)))

1 :: 2 :: 3 :: 4 :: Nil
1 :: 2 :: 3 :: 4 :: Nil


#### ex 3.14

In [20]:
fun <A> concat1(ll: List<List<A>>): List<A> =
    reverse(foldLeft(ll, List.empty<A>()) { acc, l -> foldLeft(l, acc) { acc, a -> Cons(a, acc) } })

fun <A> concat(ll: List<List<A>>): List<A> =
    foldRight(ll, List.empty<A>()) { l, acc -> foldRight(l, acc) { a, acc -> Cons(a, acc) } }

concat(List(List(1, 2), List(3, 4), List(5, 6)))

1 :: 2 :: 3 :: 4 :: 5 :: 6 :: Nil

#### ex 3.15

In [27]:
fun addOne(xs: List<Int>): List<Int> =
    when (xs) {
        is Nil -> Nil
        is Cons -> Cons(xs.head + 1, addOne(xs.tail))
        else -> error("")
    }

fun addOne1(xs: List<Int>): List<Int> {
    tailrec fun iter(xs: List<Int>, acc: List<Int>): List<Int> =
        when (xs) {
            is Nil -> acc
            is Cons -> iter(xs.tail, Cons(xs.head + 1, acc))
            else -> error("")
        }
    return iter(xs, List.empty<Int>())
}

println(addOne1(List(1, 2, 3)))

4 :: 3 :: 2 :: Nil


#### ex 3.16

In [28]:
fun doubleToStrings(xs: List<Double>): List<String> =
    when (xs) {
        is Nil -> Nil
        is Cons -> Cons(xs.head.toString(), doubleToStrings(xs.tail))
        else -> error("")
    }

println(doubleToStrings(List(1.0, 2.0, 3.0)))

1.0 :: 2.0 :: 3.0 :: Nil


#### ex 3.17

In [49]:
//fun <A, B> map(xs: List<A>, f: (A) -> B): List<B> =
//    when (xs) {
//        is Nil -> Nil
//        is Cons -> Cons(f(xs.head), map(xs.tail, f))
//        else -> error("")
//    }

fun <A, B> map(xs: List<A>, f: (A) -> B): List<B> =
    foldRight(xs, List.empty<B>()) { a, acc -> Cons(f(a), acc) }

// TODO iterative map

println(map(List(1, 2, 3)) { it + 2 })

3 :: 4 :: 5 :: Nil


#### ex 3.18

In [50]:
//fun <A> filter(xs: List<A>, pred: (A) -> Boolean): List<A> =
//    when (xs) {
//        is Nil -> Nil
//        is Cons ->
//            if (pred(xs.head)) Cons(xs.head, filter(xs.tail, pred))
//            else filter(xs.tail, pred)
//        else -> error("")
//    }

fun <A> filter(xs: List<A>, pred: (A) -> Boolean): List<A> =
    foldRight(xs, List.empty<A>()) { a, acc -> if (pred(a)) Cons(a, acc) else acc }


println(filter(List(1, 2, 3, 4, 5)) { it % 2 == 0 })

2 :: 4 :: Nil


#### ex 3.19

In [54]:
//fun <A, B> flatMap(xs: List<A>, f: (A) -> List<B>): List<B> =
//    when (xs) {
//        is Nil -> Nil
//        is Cons -> append(f(xs.head), flatMap(xs.tail, f))
//        else -> error("")
//    }

fun <A, B> flatMap(xs: List<A>, f: (A) -> List<B>): List<B> =
    foldLeft(xs, List.empty<B>()) { acc, a -> append(acc, f(a)) }

println(flatMap(List(1, 2, 3)) { List(it, it) })

1 :: 1 :: 2 :: 2 :: 3 :: 3 :: Nil


#### ex 3.20

In [33]:
fun <A> filter1(xs: List<A>, pred: (A) -> Boolean): List<A> =
    flatMap(xs) { if (pred(it)) { List(it) } else { Nil } }

println(filter1(List(1, 2, 3, 4, 5)) { it % 2 == 0 })

2 :: 4 :: Nil


#### ex 3.21

In [55]:
fun addList(xs1: List<Int>, xs2: List<Int>): List<Int> {
    return ((foldLeft(xs1, List(xs2, List.empty<Int>())) { acc, a ->
        when (acc) {
            is Nil -> error("")
            is Cons -> {
                val xs2: List<Int> = acc.head
                when (xs2) {
                    is Nil -> Nil
                    is Cons -> List(xs2.tail, Cons<Int>(a + xs2.head, (acc.tail as Cons).head))
                    else -> error("")
                }
            }
            else -> error("")
        }
    } as Cons).tail as Cons).head
}

fun addList2(xs1: List<Int>, xs2: List<Int>): List<Int> {
    val (_, res) = foldRight(xs1, reverse(xs2) to List.empty<Int>()) { a, (xs2, acc) ->
        when (xs2) {
            is Nil -> xs2 to acc
            is Cons -> xs2.tail to Cons<Int>(a + xs2.head, acc)
            else -> error("")
        }
    }
    return res
}

fun addList3(xs1: List<Int>, xs2: List<Int>): List<Int> =
    when (xs1) {
        is Nil -> Nil
        is Cons -> when (xs2) {
            is Nil -> Nil
            is Cons -> Cons(xs1.head + xs2.head, addList3(xs1.tail, xs2.tail))
            else -> error("")
        }
        else -> error("")
    }

println(addList3(List(1, 2, 3), List(3, 5, 7)))

4 :: 7 :: 10 :: Nil


In [57]:
//fun <A, B> zipWith(xs1: List<A>, xs2: List<A>, f: (A, A) -> B): List<B> {
//    val (_, res) = foldRight(xs1, reverse(xs2) to List.empty<B>()) { a, (xs2, acc) ->
//        when (xs2) {
//            is Nil -> xs2 to acc
//            is Cons -> xs2.tail to Cons<B>(f(a, xs2.head), acc)
//            else -> error("")
//        }
//    }
//    return res
//}

fun <A, B> zipWith(xs1: List<A>, xs2: List<A>, f: (A, A) -> B): List<B> =
    when (xs1) {
        is Nil -> Nil
        is Cons -> when (xs2) {
            is Nil -> Nil
            is Cons -> Cons(f(xs1.head, xs2.head), zipWith(xs1.tail, xs2.tail, f))
            else -> error("")
        }
        else -> error("")
    }

println(zipWith(List(1, 2, 3), List(3, 5, 7)) { a, b -> a + b })

4 :: 7 :: 10 :: Nil


#### ex 3.23

In [64]:
tailrec fun <A> hasSubsequence(xs: List<A>, sub: List<A>): Boolean {

    tailrec fun <A> loop(xs: List<A>, sub: List<A>): Boolean =
        when (xs) {
            is Nil -> {
                when (sub) {
                    is Nil -> true
                    else -> false
                }
            }
            is Cons -> when (sub) {
                is Nil -> true
                is Cons -> {
                    if (xs.head == sub.head)
                        loop(xs.tail, sub.tail)
                    else
                        false
                }
                else -> error("")
            }
            else -> error("")
        }

    return when (xs) {
        is Nil -> false
        is Cons -> when (sub) {
            is Nil -> false
            is Cons -> {
                if (xs.head == sub.head && loop(xs, sub)) true
                else hasSubsequence(xs.tail, sub)
            }
            else -> error("")
        }
        else -> error("")
    }
}

println(hasSubsequence(List(1, 2, 3, 4), List(2, 3)))
println(hasSubsequence(List(1, 2, 3, 4), List(2, 4)))
println(hasSubsequence(List(1, 2, 3, 4), List(2)))
println(hasSubsequence(List(1, 2, 3, 4), List(1, 2, 3, 4, 5)))
println(hasSubsequence(List(1, 2, 3, 4), List(3, 4)))
println(hasSubsequence(List(1, 2, 3, 4), List(4, 5)))

true
false
true
false
true
false


In [76]:
sealed class Tree<out A>

data class Leaf<A>(val value: A) : Tree<A>()

data class Branch<A>(val left: Tree<A>, val right: Tree<A>) : Tree<A>()

val t1: Tree<Int> = Branch(Branch(Leaf(1), Leaf(2)), Leaf(3))
val t2: Tree<Int> = Branch(Branch(Leaf(1), Leaf(20)), Branch(Leaf(3), Branch(Leaf(4), Leaf(5))))
val t3: Tree<Int> = Branch(Branch(Branch(Branch(Leaf(1), Leaf(2)), Leaf(3)), Leaf(4)), Leaf(5))

#### 3.24

In [81]:
fun <A>treesize(tree: Tree<A>): Int =
    when (tree) {
        is Leaf -> 1
        is Branch -> treesize(tree.left) + treesize(tree.right)
        else -> error("")
    }

println(treesize(t1))

3


In [84]:
fun treemaximum(tree: Tree<Int>): Int =
    when (tree) {
        is Leaf -> tree.value
        is Branch -> max(treemaximum(tree.left), treemaximum(tree.right))
        else -> error("")
    }

println(treemaximum(t2))

20


In [85]:
fun <A> treedepth(tree: Tree<A>): Int =
    when (tree) {
        is Leaf -> 1
        is Branch -> max(treedepth(tree.left), treedepth(tree.right)) + 1
        else -> error("")
    }

println(treedepth(t3))

5


In [78]:
fun <A, B>treemap(tree: Tree<A>, f: (A) -> B): Tree<B> =
    when (tree) {
        is Leaf -> Leaf(f(tree.value))
        is Branch -> Branch(treemap(tree.left, f), treemap(tree.right, f))
        else -> error("")
    }

println(treemap(t1, { it * 2}))

Branch(left=Branch(left=Leaf(value=2), right=Leaf(value=4)), right=Leaf(value=6))


In [102]:
fun <A, B> treefold(ta: Tree<A>, l: (A) -> B, b: (B, B) -> B): B =
    when (ta) {
        is Leaf -> l(ta.value)
        is Branch -> b(treefold(ta.left, l, b), treefold(ta.right, l, b))
        else -> error("")
    }

fun <A> treesizeF(ta: Tree<A>): Int =
    treefold(ta, { _ -> 1 }, { b1, b2 -> b1 + b2 })

println(treesize(t3) == treesizeF(t3))

fun treemaximumF(ta: Tree<Int>): Int =
    treefold(ta, { it }, ::max)

println(treemaximum(t2) == treemaximumF(t2))

fun <A> treedepthF(tree: Tree<A>): Int =
    treefold(tree, { 1 }, { b1, b2 -> max(b1, b2) + 1 })

println(treedepth(t3) == treedepthF(t3))

fun <A, B> treemapF(tree: Tree<A>, f: (A) -> B): Tree<B> =
    treefold(tree, { Leaf(f(it)) }, { b1: Tree<B>, b2: Tree<B> -> Branch(b1, b2) })

println(treemapF(t1, { it * 2}) == treemapF(t1, { it * 2}))


true
true
true
true
